## Milestone 3: Student Outcome Analysis
This notebook covers all analysis between employer activity and student outcomes.

**Contributors: Shaotai Hu (joining dataset, initial visualizations, documentation)**

Last Updated: 3/24/2026

Documentation link: https://docs.google.com/document/d/1noKw36_t1GgOxWHFcByYh6V7X5I12J-5jIe-ArHE_l8/edit?usp=sharing

In [ ]:
# 1. Imports & Setup
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# 2. Load Data
DATA_DIR = "/content/"   
ep_joined = pd.read_csv(os.path.join(DATA_DIR, "ep_joined.csv"))
outcomes = pd.read_csv(os.path.join(DATA_DIR, "outcomes.csv"))

# 3. Clean & Prepare Keys

# Standardize employer name fields for merging
ep_joined['employer_name_clean'] = (
    ep_joined['employer_name']
    .astype(str)
    .str.strip()
    .str.lower()
)

outcomes['Employer Name Clean'] = (
    outcomes['Employer Name']
    .astype(str)
    .str.strip()
    .str.lower()
)

# 4. Merge Without Double Counting

# Aggregate outcomes to employer level
outcomes_employer = outcomes.groupby('Employer Name Clean').agg({
    'Outcome': 'count',   
    'Employment Category': lambda x: (x == "Organization").sum(),
    'Employment Type': lambda x: (x == "Full-Time").sum()
}).rename(columns={
    'Outcome': 'num_outcomes',
    'Employment Category': 'num_org_hires',
    'Employment Type': 'num_fulltime_hires'
})

# Merge aggregated outcomes onto ep_joined
ep_outcomes = ep_joined.merge(
    outcomes_employer,
    how='left',
    left_on='employer_name_clean',
    right_on='Employer Name Clean'
)

# Fill missing values
ep_outcomes[['num_outcomes','num_org_hires','num_fulltime_hires']] = \
    ep_outcomes[['num_outcomes','num_org_hires','num_fulltime_hires']].fillna(0)

# Save new file
ep_outcomes.to_csv(os.path.join(DATA_DIR, "ep_outcomes.csv"), index=False)

# 5. Engineer New Variables

# Employer-level aggregation
employer_metrics = ep_outcomes.groupby('employer_id').agg({
    'jobs_id': 'nunique',
    'applications_count': 'sum',
    'events_name': lambda x: x.notna().sum(),
    'student_attendees_count': 'sum',
    'num_outcomes': 'sum',
    'num_fulltime_hires': 'sum'
})

employer_metrics = employer_metrics.rename(columns={
    'jobs_id': 'total_jobs',
    'applications_count': 'total_applications',
    'events_name': 'total_events',
    'student_attendees_count': 'total_attendees'
})

# Job Application Yield
employer_metrics['JAY'] = employer_metrics['total_applications'] / employer_metrics['total_jobs']

# Event Score
employer_metrics['event_score'] = employer_metrics['total_events'] + employer_metrics['total_attendees']

# Hiring Yield
employer_metrics['hiring_yield'] = employer_metrics['num_outcomes'] / employer_metrics['total_jobs']

# 6. Visualizations

# Plot 1: Event Score vs Hiring Outcomes
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=employer_metrics,
    x='event_score',
    y='num_outcomes',
    alpha=0.6
)
plt.xlabel("Event Score")
plt.ylabel("Total Student Outcomes")
plt.title("Employer Event Activity vs Student Hiring Outcomes")
plt.tight_layout()
plt.show()

# Plot 2: JAY vs Hiring Yield
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=employer_metrics,
    x='JAY',
    y='hiring_yield',
    alpha=0.6
)
plt.xlabel("Job Application Yield (JAY)")
plt.ylabel("Hiring Yield")
plt.title("Relationship Between Job Application Yield and Hiring Yield")
plt.tight_layout()
plt.show()

# Plot 3: Applications vs Outcomes
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=employer_metrics,
    x='total_applications',
    y='num_outcomes',
    alpha=0.6
)
plt.xlabel("Total Applications Received")
plt.ylabel("Total Student Outcomes")
plt.title("Do More Applications Lead to More Hires?")
plt.tight_layout()
plt.show()

# Plot 4: Correlation Heatmap
corr_cols = [
    'total_jobs','total_applications','JAY',
    'total_events','total_attendees','event_score',
    'num_outcomes','num_fulltime_hires','hiring_yield'
]

plt.figure(figsize=(10,7))
sns.heatmap(
    employer_metrics[corr_cols].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Matrix: Engagement, Applications, and Hiring Outcomes")
plt.tight_layout()
plt.show()